# PSORTb CTS Demo

End-to-end demo of the cdm_psortb CTS tool.

- **Image:** `ghcr.io/kbaseincubator/cdm_psortb:0.1.2@sha256:27ef01fb2aec9543784a6817ff6f13cee83715ae5da439182d08c0254c89b5fc`
- **Refdata:** none (PSORTb's models are bundled in the image)
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/psortb/demo`

PSORTb predicts subcellular localization of bacterial proteins. Input is **protein FASTA** (`.faa`). Required organism flag: `-n` (Gram-negative), `-p` (Gram-positive), or `-a` (Archaea). The image's wrapper script handles the underlying psort call (which only writes to `/tmp/results/<timestamp>_psortb_<organism>.txt` rather than a configurable path) and moves the result file to `/out/<input-basename>.psortb.tsv` so CTS captures it.

PSORTb processes one input FASTA per invocation, so this demo uses one container per file.

## 1. Setup

In [ ]:
tscli = get_task_service_client()
mincli = get_minio_client()

IMAGE = "ghcr.io/kbaseincubator/cdm_psortb:0.1.2@sha256:27ef01fb2aec9543784a6817ff6f13cee83715ae5da439182d08c0254c89b5fc"
OUTPUT_DIR = "cts/io/jplfaria/output/psortb/demo"

print(tscli.whoami())

## 2. List protein FASTA inputs

Use the four protein `.faa` files left over from the checkm2 demo (`cts/io/gavin/test_output/kb_demo/...`). They already have CRC64NVME checksums.

In [ ]:
input_files = []
for o in mincli.list_objects("cts", prefix="io/gavin/test_output/kb_demo", recursive=True):
    if o.object_name.endswith(".faa"):
        input_files.append(f"cts/{o.object_name}")

print(f"{len(input_files)} protein FASTA file(s):")
for f in input_files:
    print(f"  {f}")

## 3. Submit PSORTb job

Args sent to the wrapper:
- `-n`: Gram-negative organism model (use `-p` for Gram-positive, `-a` for Archaea)
- `-o terse`: tab-separated output (`SeqID\tLocalization\tScore`)
- input file via `tscli.insert_files()` (passed positionally as the last arg)

Do **not** pass `-i`, `-r`, or `-x` here. The wrapper handles input via `-i` internally and finds psort's hardcoded `/tmp/results/...` output. It moves the result to `/out/<input-basename>.psortb.tsv`.

In [ ]:
if not input_files:
    raise RuntimeError("No .faa inputs found.")

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "-n",                       # Gram-negative model
        "-o", "terse",             # tab-separated: SeqID, Localization, Score
        tscli.insert_files(),       # input file as last arg; wrapper extracts and passes via -i
    ],
    num_containers=len(input_files),
    cpus=2,
    memory="4GB",
    runtime="PT1H",
)
print("Job ID:", job.id)

## 4. Monitor (non-blocking)

In [ ]:
import threading, json
thread = threading.Thread(
    target=lambda: print(json.dumps(job.wait_for_completion(), indent=4)),
    daemon=True,
)
thread.start()

In [ ]:
# Re-run this cell to poll status
import json
print(json.dumps(job.get_job_status(), indent=2, default=str))

## 5. Inspect output files

In [ ]:
outs = job.get_job()["outputs"]
print(f"{len(outs)} output files:")
for o in outs:
    print(f"  {o['file']}")

## 6. Read PSORTb results into a dataframe

Each container writes one `.psortb.tsv` file. Columns: `SeqID`, `Localization`, `Score`. There is a header row.

In [ ]:
import io
import pandas as pd

result_files = [o for o in outs if o["file"].endswith(".psortb.tsv")]
print(f"{len(result_files)} psortb.tsv result file(s)")

frames = []
for o in result_files:
    bucket, key = o["file"].split("/", 1)
    raw = mincli.get_object(bucket, key).read().decode("utf-8", errors="replace")
    df = pd.read_csv(io.StringIO(raw), sep="\t")
    df["source_file"] = o["file"]
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nTotal rows: {len(all_df)}")
if len(all_df):
    print("\nLocalization distribution:")
    print(all_df["Localization"].value_counts())
all_df.head(20)

## 7. End-to-end check

If all of the below are True, the run was successful.

In [ ]:
checks = {
    "job complete": job.get_job_status()["state"] == "complete",
    "has .psortb.tsv files": len(result_files) > 0,
    "non-empty results": len(all_df) > 0,
    "Localization column present": "Localization" in all_df.columns if len(all_df) else False,
    "real localizations (not just header text)": (all_df["Localization"].str.len() > 0).any() if (len(all_df) and "Localization" in all_df.columns) else False,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")

if all(checks.values()):
    print("\nAll green.")
else:
    print("\nSomething's off, inspect outputs above.")